In [1]:
import warnings
warnings.filterwarnings("ignore")
import tensorflow.compat.v1 as tf
tf.disable_v2_behavior()
import numpy as np


Instructions for updating:
non-resource variables are not supported in the long term


<img src="./MNIST1.png" wigth="1400" align="left" />

MNIST 손글씨 실습을 위해서 케라스에서 제공하는 MNIST 데이터셋을 사용한다.

학습 데이터는 60,000개의 데이터가 있고 테스트 데이터는 10,000개의 데이터가 제공된다.  
MNIST 손글씨 데이터는 이미지 하나가 28개의 행과 28개의 열을 가지는 픽셀 데이터이고 각 픽셀은 흑백 사진과 같이 0부터 255까지의 그레이스케일을 가지고 있다.

In [2]:
(x_train, y_train), (x_test, y_test) = tf.keras.datasets.mnist.load_data()
print(f'x_train.shape: {x_train.shape}, y_train.shape: {y_train.shape}')
print(f'x_test.shape: {x_test.shape}, y_test.shape: {y_test.shape}')

x_train.shape: (60000, 28, 28), y_train.shape: (60000,)
x_test.shape: (10000, 28, 28), y_test.shape: (10000,)


6만개의 학습 데이터를 학습 데이터(5만개)과 검증 데이터(1만개)로 분리한다.  
학습 중간마다 검증 데이터로 모델의 성능을 측정하면 모델 학습이 제대로 진행되는지 검증 정확도를 할 수 있고 학습 정확도는 올라가는데 검증 정확도가 더 이상 올라가지 않거나 오히려 떨어질 경우 조기 종료를 구현할 수 있다.

In [3]:
# 검증 데이터로 사용하기 위해서 학습 데이터에서 1만개를 분리한다.
x_val = x_train[50000:] # 검증 데이터
x_train = x_train[:50000] # 학습 데이터
y_val = y_train[50000:] # 검증 데이터의 레이블
y_train = y_train[:50000] # 학습 데이터의 레이블
print(f'x_train.shape: {x_train.shape}, y_train.shape: {y_train.shape}')
print(f'x_val.shape: {x_val.shape}, y_val.shape: {y_val.shape}')

x_train.shape: (50000, 28, 28), y_train.shape: (50000,)
x_val.shape: (10000, 28, 28), y_val.shape: (10000,)


학습 데이터를 출력해보면 데이터가 0부터 255사이의 숫자로 구성된 것을 확인할 수 있다.

In [4]:
print(y_train[:10])
for row in x_train[0]:
    for i in row:
        print('{:3d} '.format(i), end='')
    print()

[5 0 4 1 9 2 1 3 1 4]
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0   0 
  0   0   0   0   0   0   0   0   0   0   0   0   3  18  18  18 126 136 175  26 166 255 247 127   0   0   0   0 
  0   0   0   0   0   0   0   0  30  36  94 154 170 253 253 253 253 253 225 172 253 242 195  64   0   0   0   0 
  0   0   0   0   0   0   0  49 238 253 253 253 253 253 253 253 253 251  93  82  82  56  39   0   0   0   0   0 
  0   0   0   0   0   0   0  18 219 253 253 253 253 253 198 182 247 241   

다층 퍼셉트론을 사용한 손글씨 이미지 분류 작업 흐름도

<img src="./MNIST2.png" wigth="1300" align="left" />

다층 퍼셉트론의 입력값은 무조건 1차원 형태의 배열만 가능하다.  
MNIST 손글씨 데이터는 2차원이므로 다층 퍼셉트론의 입력값으로 사용할 수 있도록 numpy의 reshape() 메소드를 사용해서 2차원 배열 형태의 데이터를 1차원 배열 형태로 변경한다.

In [5]:
print(f'x_train.shape: {x_train.shape}')
# 28행 28열로 구성된 학습 데이터를 784개의 1차원 배열 형태로 변환한다.
# x_train = np.reshape(x_train, [50000, 784])
# x_train = x_train.reshape(50000, 784)
x_train = x_train.reshape(-1, 784)
print(f'x_train.shape: {x_train.shape}')
# 28행 28열로 구성된 검증 데이터를 784개의 1차원 배열 형태로 변환한다.
x_val = x_val.reshape(-1, 784)
print(f'x_val.shape: {x_val.shape}')
# 28행 28열로 구성된 테스트 데이터를 784개의 1차원 배열 형태로 변환한다.
x_test = x_test.reshape(-1, 784)
print(f'x_test.shape: {x_test.shape}')

x_train.shape: (50000, 28, 28)
x_train.shape: (50000, 784)
x_val.shape: (10000, 784)
x_test.shape: (10000, 784)


1차원으로 변경한 데이터를 그대로 다층 퍼셉트론의 입력으로 사용해도 되지만 조금 더 효율적인 학습을 위해서 데이터를 정규화 한다.  
데이터를 정규화하면 모델의 학습 시간을 단축시키고 더 좋은 성능을 보이는 효과가 있다.  
MNIST 손글씨 데이터의 모든 값들을 0부터 255사이의 범위에 있으므로 255로 나눠서 모든 값들이 0부터 1사이의 값이되도록 정규화 한다.

In [6]:
# MNIST 손글씨 데이터 각 픽셀의 데이터 타입은 부호 없이 0부터 255사이의 값만 기억하면되므로 데이터 타입이 부호 없는 8비트 정수(uint8)로 되어있다.
print(type(x_train[0][0])) # <class 'numpy.uint8'>
# 255로 나눠서 0부터 1사이의 실수로 만들어야하므로 astype() 메소드를 사용해서 실수로 변경한다.
x_train = x_train.astype(np.float32)
print(type(x_train[0][0])) # <class 'numpy.float32'>
x_train /= 255
x_val = x_val.astype(np.float32)
x_val /= 255
x_test = x_test.astype(np.float32)
x_test /= 255

<class 'numpy.uint8'>
<class 'numpy.float32'>


MNIST 손글씨 데이터 모델은 0에서 9사이의 숫자로 분류하는 모델이므로 손실 함수로 크로스 엔트로피를 사용한다.  

In [7]:
print(set(y_train))
print(len(set(y_train)))

{np.uint8(0), np.uint8(1), np.uint8(2), np.uint8(3), np.uint8(4), np.uint8(5), np.uint8(6), np.uint8(7), np.uint8(8), np.uint8(9)}
10


크로스 엔트로피를 계산하기 위해서 레이블(y_train, y_val, y_test)을 원-핫 인코딩(one-hot encoding)으로 변경한다.  
원-핫 인코딩인 데이터를 수많은 0과 1개의 1로 구별하는 인코딩 방식으로 0으로 이루어진 벡터 집합에서 단 1개의 1의 값으로 해당 데이터를 구별하는 것을 말한다.

to_categorical() 메소드로 데이터에 원-핫 인코딩을 적용시킬 수 있다.  
to_categorical(원-핫 인코딩을 적용할 데이터, 레이블의 개수)

In [8]:
print(y_train[:5])
y_train = tf.keras.utils.to_categorical(y_train, len(set(y_train))) # 학습 데이터의 레이블에 원-핫 인코딩을 적용한다.
for i in y_train[:5]:
    print(i)
y_val = tf.keras.utils.to_categorical(y_val, len(set(y_val))) # 검증 데이터의 레이블에 원-핫 인코딩을 적용한다.
y_test = tf.keras.utils.to_categorical(y_test, len(set(y_test))) # 테스트 데이터의 레이블에 원-핫 인코딩을 적용한다.

[5 0 4 1 9]
[0. 0. 0. 0. 0. 1. 0. 0. 0. 0.]
[1. 0. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 1. 0. 0. 0. 0. 0.]
[0. 1. 0. 0. 0. 0. 0. 0. 0. 0.]
[0. 0. 0. 0. 0. 0. 0. 0. 0. 1.]


입력 데이터는 784개의 숫자가 들어있는 1차원 배열 형태이다.  
784개의 입력을 받는 256개의 노드가 첫 번째 히든 레이어에 있고 ReLU 활성화 함수를 거친 첫 번째 히든 레이어의 출력값을 입력으로 받는 두 번째 히든 레이어에는 128개의 노드가 있다.  
두 번째 히든 레이어에는 과적합을 방지하기 위해서 10%의 드롭아웃을 적용한다.  
세 번째 히든 레이어에는 총 10개의 노드가 존재하며 이 10개의 노드값은 소프트맥스를 통해서 0부터 9사이에 해당되는 각 숫자의 확률을 의미한다.  
소프트맥스는 분류해야 할 레이블의 총 개수를 k개고 할 때 k차원의 벡터를 입력받아 각 정답에 대한 확률을 계산한다.  
소프트맥스의 출력값(예측값)과 실제값의 차이를 계산하기 위해 크로스 엔트로피를 손실 함수로 사용하고 손실 함수를 최소화하기 위해서 Adam 옵티마이저(최적화 함수)를 사용해서 역전파를 통해 모든 가중치와 바이어스를 최적화 한다.

<img src="./MNIST3.png" wigth="1300" align="left" />

소프트맥스(Soft Max)의 작동 원리

In [9]:
a = tf.constant([3, 10, 5]) # 상수, 1차원 배열
sess = tf.Session()
print(sess.run(a))
# argmax() 메소드는 배열에서 가장 큰 값을 찾아서 인덱스를 리턴한다.
# a 배열에서 10이 가장 크기 때문에 결과는 10이 아니고 10의 인덱스인 1이 출력된다.
print(sess.run(tf.argmax(a)))

[ 3 10  5]
1


In [10]:
a = tf.constant([[3, 10, 5], [4, 5, 6], [0, 8, 7]]) # 상수, 2차원 배열
print(sess.run(a))
# argmax() 메소드를 2차원 배열에서 사용할 경우 적절한 2번째 인수를 지정해야 한다. 생략시 기본값은 0이다.
# 2번째 인수를 생략하거나 0을 사용하면 2차원 배열의 각 열에서 최대값의 인덱스를 리턴하고 2번째 인수로 1을 사용하면 배열의 각 행에서 최대값의 인덱스를 리턴한다.
# a 배열에서 열 단위로 최대값을 계산하면 0번째 열의 최대값은 4, 1번째 열의 최대값은 10, 2번째 열의 최대값은 7이므로 각각의 인덱스 [1 0 2]가 리턴된다.
print(sess.run(tf.argmax(a)))
print(sess.run(tf.argmax(a, 0)))
# a 배열에서 행 단위로 최대값을 계산하면 0번째 행의 최대값은 10, 1번째 행의 최대값은 6, 2번째 행의 최대값은 8이므로 각각의 인덱스 [1 2 1]가 리턴된다.
print(sess.run(tf.argmax(a, 1)))

[[ 3 10  5]
 [ 4  5  6]
 [ 0  8  7]]
[1 0 2]
[1 0 2]
[1 2 1]


다층 퍼셉트론에 사용할 placeholder를 선언한다.

placeholder를 만들때 2차원 이상의 데이터를 기억하는 경우 shape 속성을 이용해서 placeholder에 저장될 데이터의 차원을 지정해야 한다.  
X, Y placeholder shape 속성의 첫 번째 값을 None으로 지정한 이유는 데이터 개수의 제한없이 입력받기 위해서이고 X의 shape 속성 두 번째 값을 784로 지정한 이유는 MNIST 손글씨 이미지가 28 * 28 = 784 픽셀이기 때문이고 Y의 shape 속성 두 번째 인수로 10을 지정한 이유는 학습 데이터의 레이블에 to_categorical() 메소드로 원-핫 인코딩을 실행해서 레이블을 10개의 열로 만들었기 때문이다.

In [11]:
X = tf.placeholder(dtype=tf.float32, shape=[None, 784]) # 피쳐(x_train, x_val, x_test)를 기억할 placeholder
Y = tf.placeholder(dtype=tf.float32, shape=[None, 10]) # 레이블(y_train, y_val, v_test)를 기억할 placeholder
dropout = tf.placeholder(dtype=tf.float32) # 드롭아웃 값을 기억할 placeholder

다층 퍼셉트론을 구현한다.

In [12]:
# 히든 레이어 부분을 함수로 구현한다.
def mlp(x):
    # 1번째 히든 레이어
    # 784개의 입력을 받는 256개의 뉴런을 사용하기 위해서 가중치를 [입력 데이터의 개수, 뉴런의 개수] 만큼의 가중치를 난수를 만든다.
    w1 = tf.Variable(tf.random_uniform([784, 256], dtype=tf.float32))
    # 각 뉴런은 1개의 바이어스를 가진다.
    b1 = tf.Variable(tf.zeros([256]))
    # 1번째 히든 레이어의 출력, relu 활성화 함수를 사용한다.
    h1 = tf.nn.relu(tf.matmul(x, w1) + b1)
    
    # 2번째 히든 레이어
    # 1번째 히든 레이어의 출력 256개의 입력을 받는 128개의 뉴런을 사용하기 위해서 가중치를 [입력 데이터의 개수, 뉴런의 개수] 만큼의 가중치를 난수를 만든다.
    w2 = tf.Variable(tf.random_uniform([256, 128], dtype=tf.float32))
    # 각 뉴런은 1개의 바이어스를 가진다.
    b2 = tf.Variable(tf.zeros([128]))
    # 2번째 히든 레이어의 출력, relu 활성화 함수를 사용한다.
    h2 = tf.nn.relu(tf.matmul(h1, w2) + b2)
    # 2번째 히든 레이어의 출력에 드롭아웃을 적용한다.
    h2_dropout = tf.nn.dropout(h2, rate=dropout)
    
    # 3번째 히든 레이어
    # 2번째 히든 레이어의 출력 128개의 입력을 받는 10개의 뉴런을 사용하기 위해서 가중치를 [입력 데이터의 개수, 뉴런의 개수] 만큼의 가중치를 난수를 만든다.
    w3 = tf.Variable(tf.random_uniform([128, 10], dtype=tf.float32))
    # 각 뉴런은 1개의 바이어스를 가진다.
    b3 = tf.Variable(tf.zeros([10]))
    # 3번째 히든 레이어의 출력, relu 활성화 함수를 사용한다.
    # logit(LOGistic + probIT), probit는 확률을 재는 단위를 의미한다.
    logit = tf.nn.relu(tf.matmul(h2_dropout, w3) + b3)
    
    return logit

오차 함수와 옵티마이저를 적용해서 역전파를 구현한다.

In [13]:
# 히든 레이어 부분을 계산하는 함수를 호출한다.
logit = mlp(X)

# 소프트맥스를 적용한 크로스 엔트로피 손실 함수를 만든다.
loss = tf.reduce_mean(
    tf.nn.softmax_cross_entropy_with_logits_v2(logits=logit, labels=Y)
)

# Adam 옵티마이저를 사용해서 모델을 최적화 한다.
# 모델 최적화 과정은 모델의 에측값과 레이블의 차이를 줄여나가는 과정을 의미한다.
train = tf.train.AdamOptimizer(0.01).minimize(loss)

학습 시킨다.

조기 종료는 과적합을 피하는 충분한 학습을 하기 위해서 학습 중간마다 검증 데이터(x_val)에 대한 정확도를 측정하면 학습 데이터에 대한 정확도는 계속 증가하는 반면에 검증 데이터에 대한 검증 정확도가 점점 떨어질 경우 학습을 중지하는 것을 말한다.

매 epoch 마다 검증 데이터로 검증 정확도를 측정해서 검증 정확도가 5번 연속으로 검증 정확도의 최대값보다 작을 경우 조기 종료를 실행한다.

In [14]:
saver = tf.train.Saver() # tensorflow에서 학습한 모델의 저장 또는 로드에 사용할 객체를 선언한다.
epoch_cnt = 300 # 조기 종료가 일어나지 않을 경우 최대 학습 횟수를 설정한다.
batch_size = 1000 # 1번에 읽어서 처리할 학습 데이터 개수를 설정한다. 배치 크기
iteration = len(x_train) // batch_size # batch_size에 따른 1 epoch 당 학습 횟수를 설정한다.
earlystop = 5 # 현재 검증 정확도가 검증 정확도의 최대값보다 5번 연속으로 높지 않을 경우 종료하도록 설정한다.
earlystop_cnt = 0 # 현재 검증 정확도가 검증 정확도의 최대값보다 연속으로 높지 않은 횟수를 세는 변수를 선언한다.

In [16]:
with tf.Session() as sess:
    sess.run(tf.global_variables_initializer())

    prev_train_acc = 0.0 # 이전 epoch의 정확도를 기억할 변수를 선언한다.
    max_val_acc = 0.0 # 검증 정확도의 최대값을 기억할 변수를 선언한다.

    # 지정한 최대 epoch 만큼 반복하며 학습한다.
    # 현재 검증 정확도가 검증 정확도의 최대값보다 5번 연속해서 작을 경우에 조기 종료한다.
    for epoch in range(epoch_cnt):
        avg_loss = 0.0 # epoch 당 손실값을 기억할 변수를 선언한다.
        start = 0 # batch의 시작 위치
        end = batch_size # batch의 종료 위치

        # 1 epoch 학습시 학습 데이터를 batch_size개 만큼씩 나눠서 학습을 진행한다. batch 개수 만큼 반복한다.
        for i in range(iteration):
            _, _loss = sess.run([train, loss], feed_dict={X: x_train[start:end], Y: y_train[start:end], dropout: 0.1})
            # 학습할 batch 범위를 batch_size 크기 만큼 이동시킨다.
            start += batch_size
            end += batch_size
            # epoch 당 크로스 엔트로피 손실 함수의 손실값을 계산한다.
            avg_loss += _loss / iteration
        # ===== for i

        # 1 epoch 학습이 종료되면 학습 모델을 검증한다.
        # 소프트맥스를 적용해서 예측한다.
        predict = tf.nn.softmax(logits=logit)
        # 정확도를 계산하는 수식을 만든다.
        correct_predict = tf.equal(tf.argmax(predict, 1), tf.argmax(Y, 1))
        accuracy = tf.reduce_mean(tf.cast(correct_predict, dtype=tf.float32))

        # Session.run()와 Tensor.eval()의 차이
        # t가 Tensor 객체라면 t.eval()은 sess.run(t)의 속기 표현이다. 단, sess가 현재 세션인 곳에서만 가능하다.
        # 현재 학습 정확도를 계산한다.
        cur_train_acc = accuracy.eval({X: x_train, Y: y_train, dropout: 0.0})
        # 현재 검증 정확도를 계산한다.
        cur_val_acc = accuracy.eval({X: x_val, Y: y_val, dropout: 0.0})
        # epoch 당 학습 정확도 검증 정확도 출력한다.
        print('epoch: {:3d}, 현재 학습 정확도: {:7.5f}, 현재 검증 정확도: {:7.5f}'.format(epoch + 1, cur_train_acc, cur_val_acc))

        # 최대 검증 정확도와 현재 검증 정확도를 비교한다.
        if cur_val_acc <= max_val_acc:
            # 현재 검증 정확도가 최대 검증 정확도 이하이면
            # 현재 학습 정확도와 이전 학습 정확도, 현재 학습 정확도와 0.99를 비교한다.
            if cur_val_acc > prev_train_acc or cur_val_acc > 0.99:
                # 현재 학습 정확도가 이전 학습 정확도 보다 크거나 현재 학습 정확도가 0.99보다 크면
                if earlystop == earlystop_cnt:
                    print(f'조기 종료 시점: {epoch} epoch')
                    print(f'현재 검증 정확도가 최대 검증 정확도 보다 연속으로 {earlystop}번 작게 나와서 조기 종료가 실행됨')
                    break
                else:
                    # 5번 연속으로 현재 학습 정확도가 이전 학습 정확도보다 크지 않다면 이전 학습 정확도가 현재 학습 정확도 보다 크다는 것이다.
                    # 현재 검증 정확도가 최대 검증 정확도 보다 연속으로 크지 않은 횟수를 카운트 하는 변수를 1증가시키킨다.
                    earlystop_cnt += 1
                    print(f'과적합 경고 횟수: {earlystop_cnt}')
            else:
                # 현재 학습 정확도가 이전 학습 정확도 이하이고 현재 학습 정확도가 0.99 이하이면
                # 이전 학습 정확도가 현재 학습 정확도 이상이므로 현재 검증 정확도가 검증 정확도의 최대값보다 연속으로 높지않은 횟수를 카운트하는 변수를
                # 0으로 초기화시킨다.
                earlystop_cnt = 0
            # ===== if cur_val_acc > prev_train_acc
        else:
            # 현재 검증 정확도가 최대 검증 정확도 초과이면
            # 현재 검증 정확도가 최대 검증 정확도 보다 연속으로 높지 않은 횟수를 기억하는 변수를 0으로 초기화시키고 최대 검증 정확도를 현재 검증 정확도로
            # 교체한다.
            earlystop_cnt = 0
            max_val_acc = cur_val_acc
            # 검증 정확도가 가장 높은 모델을 저장한다.
            saver.save(sess, './model/model.ckpt')
        # ====== if cur_val_acc < max_val_acc
        
    # ===== for epoch
# ===== with

epoch:   1, 현재 학습 정확도: 0.30778, 현재 검증 정확도: 0.31360
epoch:   2, 현재 학습 정확도: 0.39208, 현재 검증 정확도: 0.41390
epoch:   3, 현재 학습 정확도: 0.51584, 현재 검증 정확도: 0.54110
epoch:   4, 현재 학습 정확도: 0.57948, 현재 검증 정확도: 0.60780
epoch:   5, 현재 학습 정확도: 0.62360, 현재 검증 정확도: 0.64560
epoch:   6, 현재 학습 정확도: 0.66468, 현재 검증 정확도: 0.68450
epoch:   7, 현재 학습 정확도: 0.70880, 현재 검증 정확도: 0.72550
epoch:   8, 현재 학습 정확도: 0.74222, 현재 검증 정확도: 0.76250
epoch:   9, 현재 학습 정확도: 0.76520, 현재 검증 정확도: 0.78610
epoch:  10, 현재 학습 정확도: 0.78628, 현재 검증 정확도: 0.80470
epoch:  11, 현재 학습 정확도: 0.80512, 현재 검증 정확도: 0.82440
epoch:  12, 현재 학습 정확도: 0.82030, 현재 검증 정확도: 0.83650
epoch:  13, 현재 학습 정확도: 0.83508, 현재 검증 정확도: 0.84970
epoch:  14, 현재 학습 정확도: 0.84706, 현재 검증 정확도: 0.85780
epoch:  15, 현재 학습 정확도: 0.85902, 현재 검증 정확도: 0.86800
epoch:  16, 현재 학습 정확도: 0.86884, 현재 검증 정확도: 0.87530
epoch:  17, 현재 학습 정확도: 0.87664, 현재 검증 정확도: 0.88140
epoch:  18, 현재 학습 정확도: 0.88222, 현재 검증 정확도: 0.88690
epoch:  19, 현재 학습 정확도: 0.88734, 현재 검증 정확도: 0.89280
epoch:  20, 현재 학습 정확도: 0.89312,

검증 결과가 가장 높았던 디스크에 저장된 모델을 불러온다.

In [21]:
with tf.Session() as sess:
    # 검증 정확도가 가장 높은 모델을 불러온다.
    saver.restore(sess, './model/model.ckpt')
    predict = tf.nn.softmax(logits=logit)
    correct_predict = tf.equal(tf.argmax(predict, 1), tf.argmax(Y, 1))
    accuracy = tf.reduce_mean(tf.cast(correct_predict, dtype=tf.float32))
    print('학습 정확도: {:7.5f}'.format(accuracy.eval({X: x_train, Y: y_train, dropout: 0.0})))
    print('테스트 정확도: {:7.5f}'.format(accuracy.eval({X: x_test, Y: y_test, dropout: 0.0})))

INFO:tensorflow:Restoring parameters from ./model/model.ckpt
학습 정확도: 0.95928
테스트 정확도: 0.93400
